# hybrid rarity — bench 225 AND sub-19:03

**Part 2.** The bench video established rarity for a single lift. The run video did the same for a single sport.
This model asks: what fraction of people can do both simultaneously?

Bench and run fight each other. Muscle mass helps one, slows the other. So the joint probability
is not bench_prob × run_prob — that would assume independence. It's smaller. The gap between
independence and the true joint is the **hybrid tax**.

**Foundation:** analytical bivariate normal in log-space. Exact CDF to machine precision.
MC starvation in the tails made MC unusable here — even 500k trials gave ~7 hits for 315/15:35.

**Bench counts:** using sbd_final (updated Jul 2026) — 225: 10.28M · 315: 1.18M · 405: 0.135M.
Run equivalents need to be re-matched against these counts (slightly slower than run_model used).


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import lognorm as _lognorm
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

BG    = '#0d0d0d'
PANEL = '#141414'
GRID  = '#252525'
WHITE = '#f0f0f0'
GRAY  = '#666666'
MALE  = '#4fc3f7'
FEM   = '#f48fb1'

C30 = '#00e5ff'
C25 = '#00b4ff'
C20 = '#ff2d2d'
C17 = '#ffd700'

BENCH_C225 = '#00b4ff'
BENCH_C315 = '#ff2d2d'
BENCH_C405 = '#ffd700'

RUN_MILESTONES   = [30, 25, 20, 17]
RUN_COLORS       = [C30, C25, C20, C17]
RUN_LABELS       = ['sub-30', 'sub-25', 'sub-20', 'sub-17']
BENCH_MILESTONES = [225, 315, 405]
BENCH_COLORS     = [BENCH_C225, BENCH_C315, BENCH_C405]
BENCH_LABELS     = ['225 lbs (2 plates)', '315 lbs (3 plates)', '405 lbs (4 plates)']

FRAC_MALE = 0.504
# 18-65 adults only - UN World Population Prospects 2022
# Restricts to prime training-age population, removes elderly tail
# where both bench and running performance decline sharply.
WORLD_POP = 4_100_000_000

def make_fig(w=13, h=6, cols=1, rows=1):
    if cols == 1 and rows == 1:
        fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
        _style(ax); return fig, ax
    fig, axes = plt.subplots(rows, cols, figsize=(w, h), facecolor=BG)
    for ax in np.array(axes).flatten(): _style(ax)
    return fig, axes

def _style(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=10)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    ax.grid(color=GRID, lw=0.6, alpha=0.8)

print('ready.')

In [ ]:
# ── Running distribution fits ─────────────────────────────────────────────────
# Parkrun 2023 global report percentiles (minutes)
# Male:   P10=20.5  P50=26.5  P90=37.0
# Female: P10=24.5  P50=31.5  P90=44.5

pkr_m_p10, pkr_m_med, pkr_m_p90 = 20.5, 26.5, 37.0
pkr_f_p10, pkr_f_med, pkr_f_p90 = 24.5, 31.5, 44.5

pkr_m_sig = (np.log(pkr_m_p90) - np.log(pkr_m_p10)) / (2 * stats.norm.ppf(0.90))
pkr_f_sig = (np.log(pkr_f_p90) - np.log(pkr_f_p10)) / (2 * stats.norm.ppf(0.90))

# Competitive runner - regular race entrant, trains specifically, races 4-6x/year
# Equivalent spirit to OPL: formal competition, no qualification, but self-selected serious
# Male:   P10=15.0  P50=20.5  P90=28.5   (club runner / regular race participant level)
# Female: P10=19.0  P50=25.0  P90=34.5
rce_m_p10, rce_m_med, rce_m_p90 = 15.0, 20.5, 28.5
rce_f_p10, rce_f_med, rce_f_p90 = 19.0, 25.0, 34.5

rce_m_sig = (np.log(rce_m_p90) - np.log(rce_m_p10)) / (2 * stats.norm.ppf(0.90))
rce_f_sig = (np.log(rce_f_p90) - np.log(rce_f_p10)) / (2 * stats.norm.ppf(0.90))

# ── Lifting distribution parameters ───────────────────────────────────────────
# Casual gym-goer - NSCA normative data, NA male/female
# (same values used in bench model regional adjustment)
gym_m_med, gym_m_sig = 163.0, 0.33
gym_f_med, gym_f_sig = 83.0,  0.345

# Competitive - OPL fitted values (from bench model)
opl_m_med, opl_m_sig = 282.4, 0.3126
opl_f_med, opl_f_sig = 135.0, 0.2987

print('Distribution parameters:')
print(f'  Parkrun    male  sigma={pkr_m_sig:.4f}  median={pkr_m_med} min')
print(f'  Parkrun    female sigma={pkr_f_sig:.4f}  median={pkr_f_med} min')
print(f'  Road race  male  sigma={rce_m_sig:.4f}  median={rce_m_med} min')
print(f'  Road race  female sigma={rce_f_sig:.4f}  median={rce_f_med} min')
print(f'  Gym-goer   male  sigma={gym_m_sig:.4f}  median={gym_m_med} lbs')
print(f'  Gym-goer   female sigma={gym_f_sig:.4f}  median={gym_f_med} lbs')
print(f'  OPL        male  sigma={opl_m_sig:.4f}  median={opl_m_med} lbs')
print(f'  OPL        female sigma={opl_f_sig:.4f}  median={opl_f_med} lbs')

In [ ]:
# ── Regional running participation ────────────────────────────────────────────
# Definition: adults who run regularly (3+ times/week, year-round).
# Parallel to gym-goer definition on the bench side.
# Speed distribution uses global Parkrun params (male med 26.5, female 31.5)
# as the best available proxy for regular-runner populations worldwide.
# Each region participation rate sourced and noted below.

REGIONS = [
    #  name                    pop    run%   rlo   rhi

    # Running USA 2022: ~13M US road race finishers = ~5% of adults in formal events.
    # Informal 5K runners roughly 2x race participants based on SFIA 2022 active runner
    # estimates (~27M 'core' runners out of 260M adults). Settling on 12%, was 18% -
    # the old number was closer to 'ran at any point' which is too loose.
    ('North America',          111, 0.12, 0.08, 0.17),  # 210M × 0.77 adult frac  # 210M x 0.53 (18-65 UN 2022)

    # Eurobarometer 2022: 13-15% of EU adults run regularly. UK Sport England Active
    # Lives 2022 puts running at ~15%. The 20% we had before was the high end of
    # survey ranges that don't distinguish running from walking. Using 14%.
    ('Europe',                 252, 0.14, 0.10, 0.19),  # 450M × 0.81 adult frac  # 450M x 0.56 (18-65 UN 2022)

    # Russia has a real fitness culture in major cities (Moscow, St. Petersburg) but
    # drops off hard outside urban areas. Central Asia is low. No clean survey data -
    # trimmed from 11% to 8%, treating it as a blend of EU urban and lower rural rates.
    ('Russia / C. Asia',       92, 0.08, 0.05, 0.12),  # 165M × 0.77 adult frac  # 165M x 0.56 (18-65 UN 2022)

    # China Athletics Association reported 7M+ race entrants in 2019 (up from 400K in
    # 2011) - genuine running boom in tier-1/2 cities. Japan and South Korea also have
    # strong running cultures. But this is still a thin layer over a 950M adult
    # population heavily weighted toward rural China. Keeping close to original at 11%.
    ('East Asia',              570, 0.11, 0.07, 0.16),  # 950M × 0.81 adult frac  # 950M x 0.6 (18-65 UN 2022)

    # IBGE Brazil 2022 physical activity survey: running/jogging at ~8% of active
    # adults, not total population. Concentrated in São Paulo, Rio, Bogotá, Mexico City.
    # Outside major metros it drops sharply. Cut from 12% to 7%.
    ('Latin America',          201, 0.07, 0.04, 0.11),  # 380M × 0.72 adult frac  # 380M x 0.53 (18-65 UN 2022)

    # Women's outdoor running is socially constrained in several countries across this
    # region. Extreme heat limits year-round participation broadly. GCC countries have
    # growing indoor gym culture but that doesn't translate to 5K runs. Down from 7% to 4%.
    ('Middle East / N. Af.',   161, 0.04, 0.02, 0.07),  # 310M × 0.68 adult frac  # 310M x 0.52 (18-65 UN 2022)

    # India has a visible running scene - Mumbai Marathon draws 60K+ - but it's a thin
    # urban layer over 1.4B people. WHO SEARO physical activity surveys put regular
    # exercise at 4-6% of South Asian adults; running specifically would be a fraction
    # of that. Pakistan and Bangladesh are even lower. Old 9% was way too high - dropping
    # to 3%, which might still be generous for the region as a whole.
    ('South Asia',            562, 0.03, 0.02, 0.06),  # 1080M × 0.68 adult frac  # 1080M x 0.52 (18-65 UN 2022)

    # Singapore and Malaysia have genuine running cultures (SCSM, Penang Bridge Marathon)
    # but they're small populations. Indonesia, Philippines, Vietnam drag the average
    # down - running is growing but from a very low base. Cutting from 10% to 6%.
    ('SE Asia',                275, 0.06, 0.04, 0.10),  # 500M × 0.74 adult frac  # 500M x 0.55 (18-65 UN 2022)

    # East African elites are world-famous but general population fitness running is
    # almost nonexistent - most running in the region is transport or work, not timed
    # 5Ks. Urban fitness culture is emerging in Nairobi, Lagos, Johannesburg but it's
    # a tiny fraction of 610M people. Dropping from 9% to 3%.
    ('Sub-Saharan Africa',     268, 0.03, 0.02, 0.05),  # 610M × 0.57 adult frac  # 610M x 0.44 (18-65 UN 2022)

    # Australia and NZ are genuine outliers - Parkrun thrives here, running events
    # sell out, it's culturally embedded. Running Australia surveys peg regular
    # participation at 15-18%. Small trim from 22% to 16% to stay conservative.
    ('Oceania',                 11, 0.16, 0.12, 0.21),  # 19M × 0.78 adult frac  # 19M x 0.56 (18-65 UN 2022)
]
cols = ['Region','Pop_M','RunMode','RunLo','RunHi']
df_r = pd.DataFrame(REGIONS, columns=cols)
df_r['Pop_M'] *= 1e6
df_r['Runners_M'] = df_r['Pop_M'] * df_r['RunMode'] / 1e6
print(f'Total runners (point est): {df_r["Runners_M"].sum():.0f}M')

In [ ]:
# ── Monte Carlo - 100,000 trials ──────────────────────────────────────────────
N_SIM = 100_000
R     = len(df_r)
rng   = np.random.default_rng(42)

pop = df_r['Pop_M'].values
run_s   = rng.triangular(df_r['RunLo'].values, df_r['RunMode'].values,
                          df_r['RunHi'].values, size=(N_SIM, R))
# Global Parkrun medians -- best proxy for regular-runner speed worldwide
# +-8% uncertainty to capture genuine regional variation
mmed_s  = rng.uniform(pkr_m_med * 0.92, pkr_m_med * 1.08, size=(N_SIM, 1))
fmed_s  = rng.uniform(pkr_f_med * 0.92, pkr_f_med * 1.08, size=(N_SIM, 1))
m_sig_s = rng.uniform(pkr_m_sig * 0.85, pkr_m_sig * 1.15, size=(N_SIM, 1))
f_sig_s = rng.uniform(pkr_f_sig * 0.85, pkr_f_sig * 1.15, size=(N_SIM, 1))

male_cnt   = pop * run_s * FRAC_MALE
female_cnt = pop * run_s * (1 - FRAC_MALE)

mc_results    = {}
mc_results_m  = {}
mc_results_f  = {}
mc_per_region = {}

for t in RUN_MILESTONES:
    pm = stats.norm.cdf((np.log(t) - np.log(mmed_s)) / m_sig_s)
    pf = stats.norm.cdf((np.log(t) - np.log(fmed_s)) / f_sig_s)
    mc_results[t]    = (male_cnt * pm + female_cnt * pf).sum(axis=1) / 1e6
    mc_results_m[t]  = (male_cnt * pm).sum(axis=1) / 1e6
    mc_results_f[t]  = (female_cnt * pf).sum(axis=1) / 1e6
    mc_per_region[t] = (male_cnt * pm + female_cnt * pf) / 1e6

print(f'MC complete - {N_SIM:,} trials\n')
for t in RUN_MILESTONES:
    v = mc_results[t]
    vm = mc_results_m[t]; vf = mc_results_f[t]
    print(f'  sub-{t}: {np.median(v):.1f}M total  '
          f'| male {np.median(vm):.1f}M  female {np.median(vf):.1f}M  '
          f'| [{np.percentile(v,2.5):.1f}-{np.percentile(v,97.5):.1f}M]')

In [ ]:
# Bench press world estimates from companion bench model (bench_world_model.ipynb).
# sbd_final counts (Jul 2026 refresh) — updated from bench_world_model which run_model used.
# 225: 9.27M -> 10.28M  315: 1.08M -> 1.18M  405: 0.12M -> 0.135M
# Run equivalents in PAIRS below need re-matching against these higher counts.
# Female bench 225 is tiny (0.035M) due to combined effect of lower female bench fraction
# (~20-32% of female resistance trainers do barbell bench) and lower regional female medians.
_s = 1.0
bench_results = {
    225: {'med': round(10.28*_s,2), 'lo': round(6.10*_s,2),  'hi': round(15.20*_s,2),
          'm':   round(10.22*_s,2),  'f':  round(0.04*_s,3)},
    315: {'med': round(1.18*_s,2),  'lo': round(0.37*_s,2),  'hi': round(2.65*_s,2),
          'm':   round(1.174*_s,3),  'f':  round(0.001*_s,3)},
    405: {'med': round(0.135*_s,3), 'lo': round(0.02*_s,2),  'hi': round(0.50*_s,2),
          'm':   round(0.134*_s,3),  'f':  round(0.0001*_s,4)},
}

wp  = WORLD_POP / 1e6
fig, axes = make_fig(14, 7, cols=2)

for ax, gender, glabel in [(axes[0], 'all', 'All'), (axes[1], 'm', 'Male')]:
    run_meds = [np.median({'all':mc_results,'m':mc_results_m,'f':mc_results_f}[gender][t])
                for t in RUN_MILESTONES]
    run_los  = [np.percentile({'all':mc_results,'m':mc_results_m,'f':mc_results_f}[gender][t], 2.5)
                for t in RUN_MILESTONES]
    run_his  = [np.percentile({'all':mc_results,'m':mc_results_m,'f':mc_results_f}[gender][t], 97.5)
                for t in RUN_MILESTONES]

    bk = gender if gender == 'm' else 'med'
    bench_meds = [bench_results[b][bk if bk=='m' else 'med'] for b in BENCH_MILESTONES]
    bench_los  = [bench_results[b]['lo'] for b in BENCH_MILESTONES]
    bench_his  = [bench_results[b]['hi'] for b in BENCH_MILESTONES]

    run_pcts   = [v/wp*100 for v in run_meds]
    bench_pcts = [v/wp*100 for v in bench_meds]

    ax.scatter(run_pcts,   [1]*4, color=RUN_COLORS,   s=120, zorder=5, label='Running')
    ax.scatter(bench_pcts, [0]*3, color=BENCH_COLORS, s=120, marker='D', zorder=5, label='Bench')

    for t, pct, lo, hi, c, lbl in zip(RUN_MILESTONES, run_pcts,
                                       [v/wp*100 for v in run_los],
                                       [v/wp*100 for v in run_his],
                                       RUN_COLORS, RUN_LABELS):
        ax.errorbar(pct, 1, xerr=[[pct-lo],[hi-pct]], fmt='none', color=c, lw=1.5, capsize=4)
        ax.annotate(lbl, (pct, 1), xytext=(0, 12), textcoords='offset points',
                    ha='center', color=c, fontsize=9)

    for b, pct, lo, hi, c, lbl in zip(BENCH_MILESTONES, bench_pcts,
                                        [v/wp*100 for v in bench_los],
                                        [v/wp*100 for v in bench_his],
                                        BENCH_COLORS, BENCH_LABELS):
        ax.errorbar(pct, 0, xerr=[[pct-lo],[hi-pct]], fmt='none', color=c, lw=1.5, capsize=4)
        ax.annotate(lbl, (pct, 0), xytext=(0, -18), textcoords='offset points',
                    ha='center', color=c, fontsize=9)

    ax.set_xscale('log')
    ax.set_xlim(0.0005, 5)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Bench press', 'Running'], color=WHITE, fontsize=11)
    ax.set_xlabel('% of people worldwide (log scale)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.3g}%'))
    ax.set_title(f'{glabel} - absolute worldwide rarity', color=WHITE, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
    ax.set_ylim(-0.6, 1.6)

plt.tight_layout()
plt.savefig('05_worldwide_rarity.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Analytical bivariate normal - exact joint probability ─────────────────────
# Monte Carlo starvation in the tails: 315/16:42 was getting ~7 hits out of
# 500k trials; 405/14:00 was returning 0. Estimates based on 7 samples have
# huge Poisson variance. You'd need 100M+ trials to stabilise the tail.
#
# Fix: because both marginals are log-normal, log(B) and log(R) are normal,
# so their joint distribution is exactly bivariate normal in log-space.
# scipy.stats.multivariate_normal gives the exact CDF to machine precision.
#
# P(bench >= target AND run <= target)
#   = P(run <= target) - P(bench < target AND run <= target)
#   = p_run - rv.cdf([log(bench_target), log(run_target)])

from scipy.stats import multivariate_normal

RHO_DEFAULT = 0.40

def exact_joint(bench_target, run_target,
                b_med, b_sig, r_med, r_sig,
                rho=RHO_DEFAULT):
    from scipy.stats import norm as _norm
    p_bench = 1 - _norm.cdf(np.log(bench_target), loc=np.log(b_med), scale=b_sig)
    p_run   =     _norm.cdf(np.log(run_target),   loc=np.log(r_med),  scale=r_sig)
    p_indep = p_bench * p_run

    mean = [np.log(b_med), np.log(r_med)]
    cov  = [[b_sig**2,          rho * b_sig * r_sig],
            [rho * b_sig * r_sig, r_sig**2          ]]
    rv   = multivariate_normal(mean, cov)

    p_both_below = rv.cdf([np.log(bench_target), np.log(run_target)])
    p_joint      = p_run - p_both_below

    return p_bench, p_run, p_joint, p_indep

# Run equivalents chosen by count-matching: find the run time T where the
# worldwide runner count equals the worldwide bench count at each milestone.
# TODO: re-run count-match with sbd_final counts to get updated run equivalents.
# With higher bench counts, run times will shift slower (more people can bench now).
# Placeholder times below — replace after running the inverse lookup.
# bench 225 = 10.28M -> ~sub-19:30 (estimate, needs verification)
# bench 315 = 1.18M  -> ~sub-15:45 (estimate)
# bench 405 = 0.135M -> ~sub-14:55 (estimate)
PAIRS = [
    (225, 19.05, 'bench 225 / sub-19:03', BENCH_C225),   # TODO: update
    (315, 15.58, 'bench 315 / sub-15:35', BENCH_C315),   # TODO: update
    (405, 14.83, 'bench 405 / sub-14:50', BENCH_C405),   # TODO: update
]

print(f"{'Milestone pair':<30} {'P(bench)':>9} {'P(run)':>9} "
      f"{'P(indep)':>12} {'P(joint)':>12} {'hybrid tax':>11}")
print('-' * 90)
for b_tgt, r_tgt, label, _ in PAIRS:
    pb, pr, pj, pi = exact_joint(b_tgt, r_tgt,
                                  gym_m_med, gym_m_sig,
                                  pkr_m_med, pkr_m_sig)
    tax = (pi - pj) / pi * 100 if pi > 0 else 0
    print(f"  {label:<28} {pb:>9.4%} {pr:>9.4%} {pi:>12.7%} {pj:>12.7%} {tax:>10.1f}%")

print(f"\n  rho = {RHO_DEFAULT}  |  analytical bivariate normal, exact to machine precision")


In [ ]:
# ── Hexbin scatter (Monte Carlo for visualisation only) + analytical rho sweep
# MC here is only for drawing sample points in the hexbin - not for probability
# estimates. Analytical function handles all the actual numbers.

rng = np.random.default_rng(99)
L   = np.linalg.cholesky(np.array([[1.0, RHO_DEFAULT],[RHO_DEFAULT, 1.0]]))
Z   = rng.standard_normal((2, 200_000))
U   = stats.norm.cdf(L @ Z)
bench_s = _lognorm.ppf(U[0], gym_m_sig, scale=gym_m_med)
run_s   = _lognorm.ppf(U[1], pkr_m_sig, scale=pkr_m_med)
hybrids = (bench_s >= 225) & (run_s <= 19.05)

pb, pr, pj, pi = exact_joint(225, 19.05, gym_m_med, gym_m_sig, pkr_m_med, pkr_m_sig)

fig, axes = make_fig(14, 6, cols=2)

ax = axes[0]
hb = ax.hexbin(bench_s, run_s, gridsize=60, cmap='inferno', mincnt=1,
               bins='log', extent=[30, 450, 10, 65])
ax.axvline(225,   color=BENCH_C225, lw=1.8, ls='--', alpha=0.9, label='bench 225')
ax.axhline(19.05, color=C20,        lw=1.8, ls='--', alpha=0.9, label='sub-19:03')
ax.scatter(bench_s[hybrids], run_s[hybrids],
           color=WHITE, s=2, alpha=0.25, label=f'hybrids (exact: {pj:.5%})')
ax.set_xlabel('Bench Press (lbs)'); ax.set_ylabel('5K time (minutes)')
ax.set_title('joint distribution (rho = 0.40)', color=WHITE, fontsize=12, fontweight='bold')
ax.set_xlim(30, 450); ax.set_ylim(10, 65)
ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
fig.colorbar(hb, ax=ax).ax.tick_params(colors=GRAY)

ax = axes[1]
rho_vals   = np.linspace(0.10, 0.70, 61)
joint_probs = [exact_joint(225, 19.05, gym_m_med, gym_m_sig,
                            pkr_m_med, pkr_m_sig, rho=r)[2] for r in rho_vals]
_, _, _, indep = exact_joint(225, 19.05, gym_m_med, gym_m_sig,
                              pkr_m_med, pkr_m_sig, rho=0.0)

ax.plot(rho_vals, [p * 100 for p in joint_probs], color=BENCH_C225, lw=2.5)
ax.axhline(indep * 100, color=GRAY, lw=1.5, ls=':', label=f'independent ({indep:.4%})')
ax.axvline(0.40, color=WHITE, lw=1, ls='--', alpha=0.5, label='rho=0.40 (default)')
ax.axvspan(0.35, 0.50, color=BENCH_C225, alpha=0.10, label='plausible rho range')
ax.set_xlabel('Correlation (rho)'); ax.set_ylabel('Joint probability (%)')
ax.set_title('Hybrid rarity vs assumed rho\nbench 225 + sub-19:03',
             color=WHITE, fontsize=12, fontweight='bold')
ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)

fig.suptitle('the hybrid tax',
             color=WHITE, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_hybrid_rarity.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print(f"bench 225 + sub-19:03  (rho=0.40):")
print(f"  P(bench >= 225)         = {pb:.6%}")
print(f"  P(5K  <= 19.05 min)     = {pr:.6%}")
print(f"  P(both) if independent  = {pi:.8%}")
print(f"  P(both) exact joint     = {pj:.8%}")
print(f"  Hybrid tax: {(pi-pj)/pi*100:.1f}% harder than independence assumes")


---
## next steps

1. **Re-run count-match** — scan run thresholds against sbd_final bench counts (10.28M / 1.18M / 0.135M).
   The inverse lookup is already in `run_model` — copy it here and re-run.
2. **Update PAIRS** — replace placeholder run times with verified matches.
3. **Hybrid tax chart** — grouped bars: P(bench) × P(run) vs exact joint for each milestone pair.
4. **Reel output** — 4 charts: scatter, tax, rho sweep, reveal cards.
   Target: `01_hybrid_scatter.png`, `02_hybrid_tax.png`, `03_rho_sweep.png`, `04_hybrid_reveals.png`.
